# Lezione 36 — Output strutturato

> **Modulo:** Transformer e modello open · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 22 (schema memoria), Lezione 35 (inferenza).
>
> Obiettivo pratico unico: far produrre al modello un **JSON** conforme allo
> schema Memory AI Lab e — parte pienamente eseguibile — costruire il
> **validatore/riparatore** che accetta o rifiuta l'output.
>
> ⚠️ **Nota di ambiente**: la cella che chiama Gemma e' guardata; il
> validatore gira sempre. Vedi `course/research_gaps.md`.

## Teoria essenziale

Un modello generativo produce **testo libero**: per usarlo in una pipeline serve
un formato macchina-leggibile. La strategia:

1. **Chiedere** il formato nel prompt (es. "rispondi in JSON con i campi ...").
2. **Non fidarsi**: il modello puo' sbagliare virgole, aggiungere testo, omettere
   campi. Quindi si **estrae** il blocco JSON, si **valida** contro lo schema
   (Lezione 22) e, se possibile, si **ripara**; altrimenti si rifiuta.

Il pezzo affidabile non e' il modello: e' il **validatore** attorno.

In [1]:
# Guardia di ambiente: KerasHub + pesi Gemma sono un extra opzionale (`ml`)
# e richiedono download autenticato di un modello di diversi GB. In questo
# ambiente non sono presenti, quindi le celle che usano il modello vengono
# SALTATE in modo pulito (il notebook resta eseguibile in CI). Con GPU e
# accesso a Gemma configurato, la stessa cella gira davvero.
GEMMA_AVAILABLE = False
_motivo = ""
try:
    import keras  # noqa: F401
    import keras_hub  # noqa: F401
    GEMMA_AVAILABLE = True
except Exception as exc:  # noqa: BLE001
    _motivo = f"{type(exc).__name__}: {exc}"

if GEMMA_AVAILABLE:
    print("KerasHub disponibile: le celle del modello verranno eseguite.")
else:
    print("KerasHub/Gemma NON disponibile in questo ambiente.")
    print("Motivo:", _motivo or "pacchetto assente")
    print("Le celle che richiedono il modello stampano [saltato]; "
          "il resto della lezione gira comunque.")

KerasHub/Gemma NON disponibile in questo ambiente.
Motivo: ModuleNotFoundError: No module named 'keras'
Le celle che richiedono il modello stampano [saltato]; il resto della lezione gira comunque.


In [2]:
SCHEMA_PROMPT = (
    "Return ONLY JSON with keys memory_id, text, type "
    "(one of episodic/semantic/preference). Text: "
)
if GEMMA_AVAILABLE:
    import keras_hub
    gemma = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")
    grezzo = gemma.generate(SCHEMA_PROMPT + "Marco visited Glasgow.", max_length=64)
    print(grezzo)
else:
    # esempio realistico e volutamente "sporco" (testo extra attorno al JSON)
    grezzo = ('Sure! Here is the JSON:\n'
              '{"memory_id": "mem_1", "text": "Marco visited Glasgow.", '
              '"type": "episodic"} Hope this helps!')
    print("[saltato la chiamata al modello] uso un output di esempio:")
    print(grezzo)

[saltato la chiamata al modello] uso un output di esempio:
Sure! Here is the JSON:
{"memory_id": "mem_1", "text": "Marco visited Glasgow.", "type": "episodic"} Hope this helps!


In [3]:
import json
import re

TIPI_VALIDI = {"episodic", "semantic", "preference"}
CAMPI = {"memory_id", "text", "type"}

def estrai_e_valida(grezzo):
    # 1. isola il primo blocco {...}
    m = re.search(r"\{.*\}", grezzo, re.DOTALL)
    if not m:
        return None, ["nessun blocco JSON trovato"]
    # 2. parsing
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError as e:
        return None, [f"JSON non valido: {e.msg}"]
    # 3. validazione contro lo schema (Lezione 22)
    errori = []
    mancanti = CAMPI - obj.keys()
    if mancanti:
        errori.append(f"campi mancanti: {sorted(mancanti)}")
    if obj.get("type") not in TIPI_VALIDI:
        errori.append(f"type non valido: {obj.get('type')!r}")
    return (obj if not errori else None), errori

oggetto, errori = estrai_e_valida(grezzo)
print("oggetto valido:", oggetto)
print("errori:", errori)

oggetto valido: {'memory_id': 'mem_1', 'text': 'Marco visited Glasgow.', 'type': 'episodic'}
errori: []


## Il progetto: Memory AI Lab, passo 36

Il validatore diventa il **guardiano** della pipeline: solo output conformi allo
schema entrano nel sistema di memorie. Verifico che accetti un output pulito e
rifiuti due casi rotti — tutto eseguibile, senza modello.

In [4]:
# caso valido
ok, e1 = estrai_e_valida('{"memory_id":"m1","text":"ciao","type":"semantic"}')
assert ok is not None and e1 == [], (ok, e1)
# type sbagliato
bad1, e2 = estrai_e_valida('{"memory_id":"m1","text":"ciao","type":"xyz"}')
assert bad1 is None and any("type" in x for x in e2)
# JSON malformato
bad2, e3 = estrai_e_valida('{"memory_id": "m1", "text": ')
assert bad2 is None and e3
print("validatore: accetta il valido, rifiuta type errato e JSON rotto  ✓")

validatore: accetta il valido, rifiuta type errato e JSON rotto  ✓


## Riepilogo

1. Il modello produce testo libero; la pipeline vuole **JSON**.
2. Chiedi il formato nel prompt, ma **non fidarti** dell'output.
3. Pipeline robusta: **estrai** il JSON → **valida** (schema Lezione 22) →
   ripara o **rifiuta**.
4. Il pezzo affidabile e' il **validatore**, non il modello.
5. Cosi' solo dati conformi entrano nel sistema di memorie.

## Quiz

1. Perche' non basta chiedere "rispondi in JSON"?
2. Cosa fa il validatore se manca il campo `type`?
3. Perche' isolare `{...}` con una regex prima del parsing?

*(Risposte: 1. il modello puo' aggiungere testo, sbagliare sintassi o omettere
campi; 2. segnala un errore e rifiuta l'oggetto; 3. perche' il modello spesso
avvolge il JSON in testo discorsivo che romperebbe `json.loads`.)*